# 0. Ejecutar Bronze

Propósito: Descargar los 3 datasets (SIAF, SISMEPRE, RENAMU) y convertirlos a parquet en `data/bronze/`.

> ⚠️ Hace descargas HTTP a los datastores del MEF/INEI; puede tardar varios minutos.

In [1]:
# --- Bootstrap del entorno (Windows + VSCode) ---
# VSCode inyecta el .env del repo (con rutas Linux para JAVA_HOME/HADOOP_HOME)
# dentro del kernel; en Windows esas rutas no existen y Spark no arranca.
# Ademas los notebooks corren desde notebooks/, por lo que fijamos el cwd y el
# sys.path en la raiz del repo para que 'data/...' e 'import app' funcionen.
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if not os.path.isdir(os.environ.get("JAVA_HOME", "")):
    os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot"
_hadoop = os.environ.get("HADOOP_HOME", "")
if not (os.path.isabs(_hadoop) and os.path.isdir(_hadoop)):
    os.environ["HADOOP_HOME"] = str(ROOT / "lib" / "hadoop")

print("ROOT:", ROOT)

ROOT: c:\Users\user\Downloads\EP-GDM-G6


In [2]:
from app.pipeline.bronze import BronzePipeline
from app.utils.spark import SparkClient

BronzePipeline(SparkClient()).run()
print("Bronze completado -> data/bronze/")

2026-06-19 15:30:34,765 - INFO - Iniciando Bronze Pipeline
2026-06-19 15:30:35,393 - INFO - Descargando dataset: SIAF
2026-06-19 15:30:35,395 - INFO - Descargando diccionario global para SIAF
2026-06-19 15:30:35,397 - INFO - Descargando diccionario de datos: Ingresos_diccionario
2026-06-19 15:30:36,232 - INFO - Request exitoso: 
2026-06-19 15:30:36,241 - INFO - Archivo guardado: data\bronze\dicts\Ingresos_diccionario.parquet (63 filas)
2026-06-19 15:30:36,242 - INFO - Diccionario de datos guardado: Ingresos_diccionario
2026-06-19 15:30:36,245 - INFO - Procesando módulo: 2021-Ingreso
2026-06-19 15:30:36,247 - INFO - Descargando archivo: 2021-Ingreso.zip
2026-06-19 15:30:45,572 - INFO - Request exitoso: https://fs.datosabiertos.mef.gob.pe/datastorefiles/2021-Ingreso.zip
2026-06-19 15:30:45,577 - INFO - Procesando 1 archivo(s) CSV del ZIP
2026-06-19 15:31:02,247 - INFO - Archivo guardado: data\bronze\SIAF-2021-Ingreso.parquet (828167 filas)
2026-06-19 15:31:02,307 - INFO - Módulo 2021-Ing

Bronze completado -> data/bronze/


In [3]:
# Validacion: leer el manifest y listar los parquet generados (PyArrow, sin Spark)
import pyarrow.parquet as pq
from pathlib import Path

manifest = Path("data/bronze/manifest.parquet")
if manifest.exists():
    print(pq.read_table(manifest).to_pandas().to_string(index=False))
else:
    print("No se encontro manifest.parquet")

source_name                 module_name  row_count  file_size                    downloaded_at
       SIAF                2021-Ingreso     828167    9957236 2026-06-19T20:32:20.450580+00:00
       SIAF                2022-Ingreso     774954    9681483 2026-06-19T20:32:20.650647+00:00
       SIAF                2023-Ingreso     743452    9280285 2026-06-19T20:32:20.832226+00:00
       SIAF                2024-Ingreso     758116    9399847 2026-06-19T20:32:21.017568+00:00
       SIAF                2025-Ingreso     878730   10463637 2026-06-19T20:32:21.216092+00:00
       SIAF                2026-Ingreso     354316    4608246 2026-06-19T20:32:21.413201+00:00
   SISMEPRE       rentas_ano_aplicacion         26       3885 2026-06-19T20:32:42.236687+00:00
   SISMEPRE       rentas_entidad_estado      19037      50883 2026-06-19T20:32:42.368297+00:00
   SISMEPRE rentas_esat_estadistica_atm     134170    1925885 2026-06-19T20:32:42.582831+00:00
   SISMEPRE          rentas_estadistica        233